In [103]:
import chipwhisperer as cw
import time

fw_path = "./simpleserial-ascon-CWHUSKY.hex"
scope = cw.scope()
target = cw.target(scope)

print("Found ChipWhisperer!")
scope.default_setup()
    
if hasattr(scope.io, 'target_pwr'):
    scope.io.target_pwr = 'high'
    
print(f"Programming: {fw_path}")
cw.program_target(scope, cw.programmers.SAM4SProgrammer, fw_path)

print("Resetting target...")
scope.io.nrst = 'low'
time.sleep(0.05)
scope.io.nrst = 'high'
time.sleep(0.3)
    
target.flush()

Found ChipWhisperer!
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                  

In [3]:
def calculate_r8(IV, N_128, K_128):
    # --- Constants ---
    MASK_64 = 0xFFFFFFFFFFFFFFFF
    MASK_32 = 0xFFFFFFFF
    
    N0 = N_128 & MASK_64          # Lower 64 bits
    N1 = (N_128 >> 64) & MASK_64  # Upper 64 bits
    
    K0 = K_128 & MASK_64          # Lower 64 bits
    K1 = (K_128 >> 64) & MASK_64  # Upper 64 bits
    
    iv_low = IV & MASK_32
    n1_low = N1 & MASK_32  # r5 in assembly
    k0_low = K0 & MASK_32  # sl in assembly

    r2_val = iv_low ^ n1_low

    r8_val = k0_low & (~r2_val & MASK_32)

    return hamming_weight(r8_val)

def hamming_weight(n):
    return bin(n).count('1')

In [105]:
import numpy as np
import random
from tqdm.notebook import trange

rng = np.random.default_rng()

# Configuration
N = 500000
num_samples = 1000
scope.adc.samples = num_samples
ktp = cw.ktp.Basic()
key, text = ktp.next()

# Convert key to int once
key_int = int.from_bytes(key, byteorder="little")

# Send key ONCE outside the loop
target.simpleserial_write('k', key)

# Pre-calculate constants
ASCON_128A_IV = 0x1000808c0001

# Generate all nonces upfront
nonces = rng.integers(0, 256, size=(N, 16), dtype=np.uint8)

count = 0

# Open file for writing traces
with open("ascon_cpa_dataset1.txt", "w") as f:
    for i in trange(N, desc='Capturing traces'):
        nonce = nonces[i].tobytes()
        
        # Arm scope BEFORE sending the triggering command
        scope.arm()
        
        # Send nonce (this should trigger the capture on the target)
        target.simpleserial_write('n', nonce)
        
        # Wait for capture to finish
        ret = scope.capture()
        if ret:
            print(f"Target timed out on iteration {i}!")
            continue
        
        # Get trace immediately
        trace = scope.get_last_trace()
        
        # Calculate hamming weight
        nonce_int = int.from_bytes(nonce, byteorder="little")
        hw = calculate_r8(ASCON_128A_IV, nonce_int, key_int)
        
        # Write directly to file
        row = [hw, key_int, nonce_int, *trace]
        f.write(" ".join(map(str, row)) + "\n")
        
        count += 1

print(f"Successfully captured and saved {count} traces")

Capturing traces:   0%|          | 0/500000 [00:00<?, ?it/s]

Successfully captured and saved 500000 traces


In [3]:
import numpy as np
from tqdm.notebook import trange
import sys

# Fast Helper Functions (Vectorized)
def vectorized_hamming_weight(n):
    """
    Computes Hamming Weight for a NumPy array of 32-bit integers
    using the SWAR (SIMD Within A Register) algorithm.
    This is much faster than bin().count() for arrays.
    """
    n = n - ((n >> 1) & 0x55555555)
    n = (n & 0x33333333) + ((n >> 2) & 0x33333333)
    return (((n + (n >> 4)) & 0x0F0F0F0F) * 0x01010101) >> 24

# Data Loading & Configuration
print("Loading data from file...")

MAX_TRACES = 500000 
hw_list = []
key_list = []
nonce_list = []
trace_list = []

with open("ascon_cpa_dataset.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= MAX_TRACES:
            break
        values = line.strip().split()
        hw_list.append(int(values[0]))
        key_list.append(int(values[1]))
        nonce_list.append(int(values[2]))
        trace_list.append(list(map(float, values[3:])))

print(f"Loaded {len(hw_list)} traces")

# Convert to numpy arrays
hw_array = np.array(hw_list, dtype=np.int32)
key_array = np.array(key_list, dtype=object)
nonce_array = np.array(nonce_list, dtype=object)
trace_array = np.array(trace_list, dtype=np.float32)

num_traces = len(trace_array)
num_samples = trace_array.shape[1]

print(f"Number of traces: {num_traces}")
print(f"Samples per trace: {num_samples}")

# Constants
ASCON_128A_IV = 0x1000808c0001
MASK_32 = 0xFFFFFFFF
MASK_64 = 0xFFFFFFFFFFFFFFFF

print("\n1. Pre-computing Trace Statistics...")

t_bar = np.mean(trace_array, axis=0, dtype=np.float64)
t_centered = trace_array.astype(np.float64) - t_bar
o_t = np.sqrt(np.sum(t_centered**2, axis=0))

print("2. Pre-processing Nonces for Ascon Logic...")

n1_low_vector = ((nonce_array >> 64) & MASK_32).astype(np.uint32)
n1_low_vector = n1_low_vector.reshape(-1, 1)
iv_low = np.uint32(ASCON_128A_IV & MASK_32)

cparefs = [0.0] * 4
bestguess = [0] * 4

print("\nStarting Optimized CPA attack on K0_low (4 bytes)...")

for bnum in range(4):
    print(f"--> Attacking Byte {bnum}...")

    k_guess_vals = np.arange(256, dtype=np.uint32).reshape(1, -1)
    k0_low_guesses = k_guess_vals << (8 * bnum)
    r2_val = iv_low ^ n1_low_vector
    r8_val = k0_low_guesses & (~r2_val & MASK_32)
    
    hws = vectorized_hamming_weight(r8_val).astype(np.float64)
    
    h_bar = np.mean(hws, axis=0) 
    h_centered = hws - h_bar
    o_h = np.sqrt(np.sum(h_centered**2, axis=0))
    numerator = np.dot(h_centered.T, t_centered)
    denominator = o_h[:, None] * o_t[None, :]
    
    # Avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        cpa_matrix = numerator / denominator
        cpa_matrix = np.nan_to_num(cpa_matrix)
    
    # Find Best Candidate
    
    # Get max correlation for each key guess
    max_cpa_per_guess = np.max(np.abs(cpa_matrix), axis=1)  # Shape (256,)
    
    # The guess with the highest peak is our winner
    best_k = np.argmax(max_cpa_per_guess)
    max_corr = max_cpa_per_guess[best_k]
    
    bestguess[bnum] = best_k
    cparefs[bnum] = float(max_corr)
    
    print(f"    Best guess: 0x{best_k:02x} (Corr: {max_corr:.4f})")
    
    # Clean up large matrices to free RAM for next loop
    del hws, h_centered, numerator, cpa_matrix, r8_val

# Results & Verification
print("\n" + "="*60)
print("ATTACK RESULTS")
print("="*60)

print("Best K0_low Guess (4 bytes): ", end="")
for b in bestguess:
    print("%02x " % b, end="")
print()

# Reconstruct full K0_low
k0_low_recovered = sum(bestguess[i] << (8*i) for i in range(4))
print(f"\nRecovered K0_low: 0x{k0_low_recovered:08x}")

# Verification
actual_key = int(key_array[0])
K0_actual = actual_key & MASK_64
k0_low_actual = K0_actual & MASK_32
print(f"Actual K0_low:    0x{k0_low_actual:08x}")

if k0_low_recovered == k0_low_actual:
    print("\n✓ SUCCESS! Key recovered correctly!")
else:
    print("\n✗ Key recovery failed.")
    print("  Try increasing MAX_TRACES or checking trigger alignment")

print("\nCorrelation coefficients:", cparefs)

Loading data from file...
Loaded 500000 traces
Number of traces: 500000
Samples per trace: 1000
Memory usage: ~1907.3 MB for traces

1. Pre-computing Trace Statistics...
2. Pre-processing Nonces for Ascon Logic...

Starting Optimized CPA attack on K0_low (4 bytes)...
--> Attacking Byte 0...
    Best guess: 0xf7 (Corr: 0.4220)
--> Attacking Byte 1...
    Best guess: 0x7e (Corr: 0.4296)
--> Attacking Byte 2...
    Best guess: 0x73 (Corr: 0.3775)
--> Attacking Byte 3...
    Best guess: 0x5f (Corr: 0.4181)

ATTACK RESULTS
Best K0_low Guess (4 bytes): f7 7e 73 5f 

Recovered K0_low: 0x5f737ef7
Actual K0_low:    0x16157e2b

✗ Key recovery failed.
  Try increasing MAX_TRACES or checking trigger alignment

Correlation coefficients: [0.4220126825580165, 0.42959774450576554, 0.3775467993245002, 0.418089737153865]


In [3]:
import numpy as np
from tqdm.notebook import trange
import sys

def vectorized_hamming_weight(n):
    n = n - ((n >> 1) & 0x55555555)
    n = (n & 0x33333333) + ((n >> 2) & 0x33333333)
    return (((n + (n >> 4)) & 0x0F0F0F0F) * 0x01010101) >> 24

print("Loading data from file...")

MAX_TRACES = 15000

hw_list = []
key_list = []
nonce_list = []
trace_list = []

with open("ascon_cpa_dataset.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= MAX_TRACES:
            break
        values = line.strip().split()
        hw_list.append(int(values[0]))
        key_list.append(int(values[1]))
        nonce_list.append(int(values[2]))
        trace_list.append(list(map(float, values[3:])))

print(f"Loaded {len(hw_list)} traces")

# Convert to numpy arrays
hw_array = np.array(hw_list, dtype=np.int32)
key_array = np.array(key_list, dtype=object)
nonce_array = np.array(nonce_list, dtype=object)
trace_array = np.array(trace_list, dtype=np.float32)

num_traces = len(trace_array)
num_samples = trace_array.shape[1]

print(f"Number of traces: {num_traces}")
print(f"Samples per trace: {num_samples}")

# Constants
ASCON_128A_IV = 0x1000808c0001
MASK_32 = 0xFFFFFFFF
MASK_64 = 0xFFFFFFFFFFFFFFFF

print("\n1. Pre-computing Trace Statistics...")

t_bar = np.mean(trace_array, axis=0, dtype=np.float64)
t_centered = trace_array.astype(np.float64) - t_bar
o_t = np.sqrt(np.sum(t_centered**2, axis=0))

print("2. Pre-processing Nonces for Ascon Logic...")

n1_low_vector = ((nonce_array >> 64) & MASK_32).astype(np.uint32)
n1_low_vector = n1_low_vector.reshape(-1, 1)
iv_low = np.uint32(ASCON_128A_IV & MASK_32)

cparefs = [0.0] * 2
bestguess = [0] * 2
top10_results = []
recovered_key_state = 0  # <--- NEW: accumulates the key as we find it

print("\nStarting Optimized CPA attack on K0_low (Iterative)...")

for bnum in range(2): 
    print(f"\n{'='*60}")
    print(f"Attacking Bytes {bnum*2} and {bnum*2+1} (16-bit guess)")
    print(f"{'='*60}")

    k_guess_16bit = np.arange(65536, dtype=np.uint32).reshape(1, -1)
    
    shift_amount = 16 * bnum
    
    k0_low_guesses = (k_guess_16bit << shift_amount) | recovered_key_state

    r2_val = iv_low ^ n1_low_vector  # (N, 1)
    
    r8_val = k0_low_guesses & (~r2_val & MASK_32)
    
    # 4. Calculate Hamming Weight
    print(f"Computing {r8_val.shape[1]} hypotheses...")
    hws = vectorized_hamming_weight(r8_val).astype(np.float64)
    
    print(f"Computing correlations...")
    h_bar = np.mean(hws, axis=0) 
    h_centered = hws - h_bar
    o_h = np.sqrt(np.sum(h_centered**2, axis=0))
    
    numerator = np.dot(h_centered.T, t_centered)
    denominator = o_h[:, None] * o_t[None, :]
    
    with np.errstate(divide='ignore', invalid='ignore'):
        cpa_matrix = numerator / denominator
        cpa_matrix = np.nan_to_num(cpa_matrix)
    
    max_cpa_per_guess = np.max(np.abs(cpa_matrix), axis=1)
    
    # Get top 10
    top10_indices = np.argsort(max_cpa_per_guess)[::-1][:10]
    
    segment_top10 = []
    for rank, idx in enumerate(top10_indices, 1):
        corr = max_cpa_per_guess[idx]
        segment_top10.append({
            'rank': rank, 'key_guess': idx, 'correlation': corr
        })
    top10_results.append(segment_top10)
    
    # Winner for this round
    best_k = top10_indices[0]
    max_corr = max_cpa_per_guess[best_k]
    
    bestguess[bnum] = best_k
    cparefs[bnum] = float(max_corr)
    
    recovered_key_state |= (best_k << shift_amount)
    print(f"Current recovered state: 0x{recovered_key_state:08x}")

    # Display Top 10
    print(f"\nTop 10 Candidates for Bytes {bnum*2}-{bnum*2+1}:")
    print(f"{'Rank':<6}{'Key Guess':<12}{'Correlation':<15}")
    print("-" * 40)
    for result in segment_top10:
        print(f"{result['rank']:<6}0x{result['key_guess']:04x}{' '*6}{result['correlation']:.6f}")
    
    del hws, h_centered, numerator, cpa_matrix, r8_val

# ============================================================
# 5. Results & Verification
# ============================================================
print("\n" + "="*60)
print("FINAL ATTACK RESULTS")
print("="*60)

print("\nBest K0_low Guess (2 x 16-bit):")
print(f"  Bytes 0-1: 0x{bestguess[0]:04x} (Correlation: {cparefs[0]:.6f})")
print(f"  Bytes 2-3: 0x{bestguess[1]:04x} (Correlation: {cparefs[1]:.6f})")

# Reconstruct full K0_low from two 16-bit guesses
k0_low_recovered = bestguess[0] | (bestguess[1] << 16)
print(f"\nRecovered K0_low: 0x{k0_low_recovered:08x}")

# Break down into individual bytes for clarity
bytes_recovered = [
    k0_low_recovered & 0xFF,
    (k0_low_recovered >> 8) & 0xFF,
    (k0_low_recovered >> 16) & 0xFF,
    (k0_low_recovered >> 24) & 0xFF
]
print(f"As individual bytes: {' '.join(f'{b:02x}' for b in bytes_recovered)}")

# Verification
actual_key = int(key_array[0])
K0_actual = actual_key & MASK_64
k0_low_actual = K0_actual & MASK_32
print(f"\nActual K0_low:    0x{k0_low_actual:08x}")

if k0_low_recovered == k0_low_actual:
    print("\n✓ SUCCESS! Key recovered correctly!")
else:
    print("\n✗ Key recovery failed.")
    bytes_actual = [
        k0_low_actual & 0xFF,
        (k0_low_actual >> 8) & 0xFF,
        (k0_low_actual >> 16) & 0xFF,
        (k0_low_actual >> 24) & 0xFF
    ]
    print(f"Actual bytes:     {' '.join(f'{b:02x}' for b in bytes_actual)}")

# Check if correct key is in top 10 for each segment
print("\n" + "="*60)
print("TOP 10 ANALYSIS")
print("="*60)

for bnum in range(2):
    actual_segment = (k0_low_actual >> (16 * bnum)) & 0xFFFF
    print(f"\nBytes {bnum*2}-{bnum*2+1}: Actual value = 0x{actual_segment:04x}")
    
    found = False
    for result in top10_results[bnum]:
        if result['key_guess'] == actual_segment:
            print(f"  ✓ Correct key found at rank {result['rank']} "
                  f"(Correlation: {result['correlation']:.6f})")
            found = True
            break
    
    if not found:
        print(f"  ✗ Correct key NOT in top 10")
        print(f"  Top guess was: 0x{top10_results[bnum][0]['key_guess']:04x} "
              f"(Correlation: {top10_results[bnum][0]['correlation']:.6f})")

Loading data from file...
Loaded 15000 traces
Number of traces: 15000
Samples per trace: 1000

1. Pre-computing Trace Statistics...
2. Pre-processing Nonces for Ascon Logic...

Starting Optimized CPA attack on K0_low (Iterative)...

Attacking Bytes 0 and 1 (16-bit guess)
Computing 65536 hypotheses...
Computing correlations...
Current recovered state: 0x00007e2b

Top 10 Candidates for Bytes 0-1:
Rank  Key Guess   Correlation    
----------------------------------------
1     0x7e2b      0.520455
2     0x7e2a      0.504724
3     0x7e29      0.503822
4     0x6e2b      0.503502
5     0xeec6      0.502487
6     0x7e23      0.499363
7     0x3e2b      0.497228
8     0x30f7      0.496682
9     0x7e0b      0.492916
10    0x762b      0.489850

Attacking Bytes 2 and 3 (16-bit guess)
Computing 65536 hypotheses...
Computing correlations...
Current recovered state: 0x16157e2b

Top 10 Candidates for Bytes 2-3:
Rank  Key Guess   Correlation    
----------------------------------------
1     0x1615    

In [5]:

import numpy as np
from tqdm.notebook import trange
import sys

def vectorized_hamming_weight(n):
    n = n - ((n >> 1) & 0x55555555)
    n = (n & 0x33333333) + ((n >> 2) & 0x33333333)
    return (((n + (n >> 4)) & 0x0F0F0F0F) * 0x01010101) >> 24

print("Loading data from file...")

MAX_TRACES = 15000  # Adjust based on your memory

hw_list = []
key_list = []
nonce_list = []
trace_list = []

with open("ascon_cpa_dataset.txt", "r") as f:
    for i, line in enumerate(f):
        if i >= MAX_TRACES:
            break
        values = line.strip().split()
        hw_list.append(int(values[0]))
        key_list.append(int(values[1]))
        nonce_list.append(int(values[2]))
        trace_list.append(list(map(float, values[3:])))

print(f"Loaded {len(hw_list)} traces")

# Convert to numpy arrays
hw_array = np.array(hw_list, dtype=np.int32)
key_array = np.array(key_list, dtype=object)
nonce_array = np.array(nonce_list, dtype=object)
trace_array = np.array(trace_list, dtype=np.float32)

num_traces = len(trace_array)
num_samples = trace_array.shape[1]

print(f"Number of traces: {num_traces}")
print(f"Samples per trace: {num_samples}")

# Constants
ASCON_128A_IV = 0x1000808c0001
MASK_32 = 0xFFFFFFFF
MASK_64 = 0xFFFFFFFFFFFFFFFF

print("\n1. Pre-computing Trace Statistics...")

t_bar = np.mean(trace_array, axis=0, dtype=np.float64)
t_centered = trace_array.astype(np.float64) - t_bar
o_t = np.sqrt(np.sum(t_centered**2, axis=0))

print("2. Pre-processing Nonces for Ascon Logic...")

# Extract HIGH 32 bits of the nonce's lower 64 bits
n1_high_vector = ((nonce_array >> 32) & MASK_32).astype(np.uint32)
n1_high_vector = n1_high_vector.reshape(-1, 1)

# Extract HIGH 32 bits of the IV
iv_high = np.uint32((ASCON_128A_IV >> 32) & MASK_32)

cparefs = [0.0] * 2
bestguess = [0] * 2
top10_results = []
recovered_key_state = 0  # Accumulates the key as we find it

print("\nStarting Optimized CPA attack on K0_high (Iterative)...")

for bnum in range(2): 
    print(f"\n{'='*60}")
    print(f"Attacking Bytes {bnum*2} and {bnum*2+1} (16-bit guess)")
    print(f"{'='*60}")

    k_guess_16bit = np.arange(65536, dtype=np.uint32).reshape(1, -1)
    
    shift_amount = 16 * bnum
    
    k0_high_guesses = (k_guess_16bit << shift_amount) | recovered_key_state

    r2_val = iv_high ^ n1_high_vector  # (N, 1)
    
    r8_val = k0_high_guesses & (~r2_val & MASK_32)
    
    # 4. Calculate Hamming Weight
    print(f"Computing {r8_val.shape[1]} hypotheses...")
    hws = vectorized_hamming_weight(r8_val).astype(np.float64)
    
    print(f"Computing correlations...")
    h_bar = np.mean(hws, axis=0) 
    h_centered = hws - h_bar
    o_h = np.sqrt(np.sum(h_centered**2, axis=0))
    
    numerator = np.dot(h_centered.T, t_centered)
    denominator = o_h[:, None] * o_t[None, :]
    
    with np.errstate(divide='ignore', invalid='ignore'):
        cpa_matrix = numerator / denominator
        cpa_matrix = np.nan_to_num(cpa_matrix)
    
    max_cpa_per_guess = np.max(np.abs(cpa_matrix), axis=1)
    
    # Get top 10
    top10_indices = np.argsort(max_cpa_per_guess)[::-1][:10]
    
    segment_top10 = []
    for rank, idx in enumerate(top10_indices, 1):
        corr = max_cpa_per_guess[idx]
        segment_top10.append({
            'rank': rank, 'key_guess': idx, 'correlation': corr
        })
    top10_results.append(segment_top10)
    
    # Winner for this round
    best_k = top10_indices[0]
    max_corr = max_cpa_per_guess[best_k]
    
    bestguess[bnum] = best_k
    cparefs[bnum] = float(max_corr)
    
    recovered_key_state |= (best_k << shift_amount)
    print(f"Current recovered state: 0x{recovered_key_state:08x}")

    # Display Top 10
    print(f"\nTop 10 Candidates for Bytes {bnum*2}-{bnum*2+1}:")
    print(f"{'Rank':<6}{'Key Guess':<12}{'Correlation':<15}")
    print("-" * 40)
    for result in segment_top10:
        print(f"{result['rank']:<6}0x{result['key_guess']:04x}{' '*6}{result['correlation']:.6f}")
    
    del hws, h_centered, numerator, cpa_matrix, r8_val

# ============================================================
# 5. Results & Verification
# ============================================================
print("\n" + "="*60)
print("FINAL ATTACK RESULTS")
print("="*60)

print("\nBest K0_high Guess (2 x 16-bit):")
print(f"  Bytes 0-1: 0x{bestguess[0]:04x} (Correlation: {cparefs[0]:.6f})")
print(f"  Bytes 2-3: 0x{bestguess[1]:04x} (Correlation: {cparefs[1]:.6f})")

# Reconstruct full K0_high from two 16-bit guesses
k0_high_recovered = bestguess[0] | (bestguess[1] << 16)
print(f"\nRecovered K0_high: 0x{k0_high_recovered:08x}")

# Break down into individual bytes for clarity
bytes_recovered = [
    k0_high_recovered & 0xFF,
    (k0_high_recovered >> 8) & 0xFF,
    (k0_high_recovered >> 16) & 0xFF,
    (k0_high_recovered >> 24) & 0xFF
]
print(f"As individual bytes: {' '.join(f'{b:02x}' for b in bytes_recovered)}")

# Verification
actual_key = int(key_array[0])
K0_actual = actual_key & MASK_64
k0_high_actual = (K0_actual >> 32) & MASK_32
print(f"\nActual K0_high:   0x{k0_high_actual:08x}")

if k0_high_recovered == k0_high_actual:
    print("\n✓ SUCCESS! Key recovered correctly!")
else:
    print("\n✗ Key recovery failed.")
    bytes_actual = [
        k0_high_actual & 0xFF,
        (k0_high_actual >> 8) & 0xFF,
        (k0_high_actual >> 16) & 0xFF,
        (k0_high_actual >> 24) & 0xFF
    ]
    print(f"Actual bytes:     {' '.join(f'{b:02x}' for b in bytes_actual)}")

# Check if correct key is in top 10 for each segment
print("\n" + "="*60)
print("TOP 10 ANALYSIS")
print("="*60)

for bnum in range(2):
    actual_segment = (k0_high_actual >> (16 * bnum)) & 0xFFFF
    print(f"\nBytes {bnum*2}-{bnum*2+1}: Actual value = 0x{actual_segment:04x}")
    
    found = False
    for result in top10_results[bnum]:
        if result['key_guess'] == actual_segment:
            print(f"  ✓ Correct key found at rank {result['rank']} "
                  f"(Correlation: {result['correlation']:.6f})")
            found = True
            break
    
    if not found:
        print(f"  ✗ Correct key NOT in top 10")
        print(f"  Top guess was: 0x{top10_results[bnum][0]['key_guess']:04x} "
              f"(Correlation: {top10_results[bnum][0]['correlation']:.6f})")


Starting Optimized CPA attack on K0_high (Corrected Model)...

Attacking Bytes 4 and 5 (High Word)
Computing 65536 hypotheses...
Computing correlations...

Top 10 Candidates for High Bytes 0-1:
Rank  Key Guess   Correlation    
----------------------------------------
1     0x0000      0.857971
2     0x0100      0.846663
3     0x0001      0.846483
4     0x0040      0.846234
5     0x0020      0.845785
6     0x0004      0.845236
7     0x0080      0.844413
8     0x0002      0.844195
9     0x0800      0.844165
10    0x0008      0.844094

Attacking Bytes 6 and 7 (High Word)
Computing 65536 hypotheses...
Computing correlations...

Top 10 Candidates for High Bytes 2-3:
Rank  Key Guess   Correlation    
----------------------------------------
1     0x0000      0.857971
2     0x0020      0.846953
3     0x4000      0.846805
4     0x0040      0.846352
5     0x0400      0.846175
6     0x0100      0.845626
7     0x0008      0.845381
8     0x1000      0.845198
9     0x0004      0.845099
10    0x00

In [43]:

for batch in range(1000):
    print(f"Batch {batch}")
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    N = 1000
    num_samples = 500
    scope.adc.samples = num_samples
    key_int = int.from_bytes(key, byteorder="little")

    trace_array = np.empty((N, num_samples), dtype=np.float32)
    nonce_array = np.empty((N, 16), dtype=np.uint8)
    hamming_weights = np.empty(N, dtype=np.uint32)

    nonces = rng.integers(0, 256, size=(N, 16), dtype=np.uint8)

    count = 0
    for i in trange(N, desc='Capturing traces'):
        nonce = nonces[i].tobytes()
        
        # Arm scope BEFORE sending the triggering command
        scope.arm()
        
        # Send nonce (this should trigger the capture on the target)
        target.simpleserial_write('n', nonce)
        
        # Wait for capture to finish (hardware wait is faster than sleep)
        ret = scope.capture()
        if ret:
            print(f"Target timed out on iteration {i}!")
            continue
    
        # Fast storage
        trace_array[count] = scope.get_last_trace()
        nonce_array[count] = bytearray(nonce)
        count += 1 

    for i in range(count):
        hamming_weights[i] = calculate_r8(
            ASCON_128A_IV,
            int.from_bytes(nonce_array[i], "little"),
            key_int
        )
    trace_array = trace_array[:count]
    nonce_array = nonce_array[:count]
    hamming_weights = hamming_weights[:count]
    traces = trace_array
    hw = hamming_weights.astype(np.float64)

    # Center
    traces_centered = traces - traces.mean(axis=0)   # (N, T)
    hw_centered = hw - hw.mean()                     # (N,)

    # Pearson correlation
    numerator = np.dot(hw_centered, traces_centered)  # (T,)
    denominator = np.sqrt(
        np.sum(hw_centered ** 2) *
        np.sum(traces_centered ** 2, axis=0)
    )

    correlation = numerator / denominator

    print("Correlation array shape:", correlation.shape)  # (num_samples,)
    plt.figure(figsize=(12,4))
    plt.plot(correlation)
    plt.title("CPA: Pearson correlation coefficient vs time")
    plt.xlabel("Sample index")
    plt.ylabel("Correlation coefficient")
    plt.grid(True)
    plt.show()
        

Batch 0


Capturing traces:   0%|          | 0/1000 [00:00<?, ?it/s]

USBErrorTimeout: LIBUSB_ERROR_TIMEOUT [-7]

In [50]:
import numpy as np

# Load entire file
data = np.loadtxt("ascon_cpa_dataset.txt")

# Split columns
hw     = data[:, 0]        # shape (N,)
keys   = data[:, 1]        # shape (N,)
nonces = data[:, 2]        # shape (N,)
traces = data[:, 3:]       # shape (N, T)

print("Traces shape:", traces.shape)
print("HW shape:", hw.shape)

Traces shape: (1000, 1000)
HW shape: (1000,)
